In [0]:
# Análise Exploratória de Dados (EDA)
# Squad 1 - Dupla 3
# Tabelas: ecommerce_produtos e ecommerce_categorias

In [0]:
%pip install azure-storage-file-datalake azure-identity pandas pyarrow
dbutils.library.restartPython()

In [0]:
import pandas as pd

In [0]:
tabela_produtos = "ecommerce_produtos"
tabela_categorias = "ecommerce_categorias"

print("Tabela 1:", tabela_produtos)
print("Tabela 2:", tabela_categorias)

In [0]:
df_produtos = None
df_categorias = None

print("DataFrames preparados para leitura dos dados.")

In [0]:
storage_account_name = "internshipdatalake"
container_name = "raw"

adls_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net/"
)

caminho_real_time = f"{adls_path}real-time-data/"

print("Diretório de referência:")
print(caminho_real_time)

In [0]:
from dotenv import load_dotenv
import os
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
import pandas as pd
from io import BytesIO

load_dotenv(
    "/Workspace/Repos/diniz.dbruna@gmail.com/estagio-empregadados-turma-2/.env",
    override=True,
)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential,
)
fs_client = service_client.get_file_system_client(container_name)

# Encontra a pasta de timestamp mais recente dentro de real-time-data
def encontrar_pasta_mais_recente(fs_client, base_path="real-time-data"):
    pastas = set()
    for p in fs_client.get_paths(path=base_path, recursive=True):
        if not p.is_directory:
            pasta = "/".join(p.name.split("/")[:-1])
            pastas.add(pasta)
    return sorted(pastas)[-1]

pasta_mais_recente = encontrar_pasta_mais_recente(fs_client)
print("Lendo a partir de:", pasta_mais_recente)

def ler_parquet_do_adls(caminho_no_container):
    file_client = fs_client.get_file_client(caminho_no_container)
    download = file_client.download_file()
    conteudo = download.readall()
    return pd.read_parquet(BytesIO(conteudo))

df_produtos_pd = ler_parquet_do_adls(f"{pasta_mais_recente}/ecommerce_produtos.parquet")
df_categorias_pd = ler_parquet_do_adls(f"{pasta_mais_recente}/ecommerce_categorias.parquet")

df_produtos = spark.createDataFrame(df_produtos_pd)
df_categorias = spark.createDataFrame(df_categorias_pd)

print("Produtos:", df_produtos.count(), "linhas")
print("Categorias:", df_categorias.count(), "linhas")

In [0]:
from pyspark.sql import functions as F

def analisar_qualidade_dataframe(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Qualidade dos dados: {nome} ===")

    print("\nValores nulos por coluna:")
    df.select([
        F.sum(F.col(col).isNull().cast("int")).alias(col)
        for col in df.columns
    ]).show()

    total_registros = df.count()
    registros_unicos = df.dropDuplicates().count()
    duplicados = total_registros - registros_unicos

    print("\nDuplicidades:")
    print(f"Total de registros: {total_registros}")
    print(f"Registros duplicados: {duplicados}")

In [0]:
def analisar_schema_dataframe(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Schema: {nome} ===")
    df.printSchema()

In [0]:
def analisar_estatisticas_dataframe(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Estatísticas descritivas: {nome} ===")
    df.describe().show()

In [0]:
def analisar_valores_distintos(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    print(f"=== Valores distintos: {nome} ===")

    for coluna in df.columns:
        quantidade = df.select(coluna).distinct().count()
        print(f"{coluna}: {quantidade} valores distintos")

In [0]:
def analisar_duplicidades_coluna(df, coluna, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    if coluna not in df.columns:
        print(f"{nome}: a coluna '{coluna}' não existe.")
        return

    total = df.count()
    distintos = df.select(coluna).distinct().count()
    duplicados = total - distintos

    print(f"=== Duplicidades: {nome} ===")
    print(f"Coluna analisada: {coluna}")
    print(f"Total de registros: {total}")
    print(f"Valores distintos: {distintos}")
    print(f"Possíveis duplicidades: {duplicados}")

In [0]:
def analisar_registros_nulos(df, nome):
    if df is None:
        print(f"{nome}: dados ainda não carregados.")
        return

    condicao = None

    for coluna in df.columns:
        expressao = F.col(coluna).isNotNull()

        if condicao is None:
            condicao = expressao
        else:
            condicao = condicao | expressao

    registros_nulos = df.filter(~condicao).count()

    print(f"=== Registros completamente nulos: {nome} ===")
    print(f"Registros completamente nulos: {registros_nulos}")

In [0]:
for nome, df in [("Produtos", df_produtos), ("Categorias", df_categorias)]:
    analisar_qualidade_dataframe(df, nome)
    analisar_schema_dataframe(df, nome)
    analisar_estatisticas_dataframe(df, nome)
    analisar_valores_distintos(df, nome)
    analisar_registros_nulos(df, nome)

# analisar_duplicidades_coluna precisa do nome de uma coluna específica.
# Ajuste para uma coluna real de identificação depois de ver o schema acima:
# analisar_duplicidades_coluna(df_produtos, "produto_id", "Produtos")
# analisar_duplicidades_coluna(df_categorias, "categoria_id", "Categorias")

In [0]:
analisar_duplicidades_coluna(df_produtos, "sku", "Produtos")
analisar_duplicidades_coluna(df_categorias, "id_categoria", "Categorias")

In [0]:
df_produtos.join(df_categorias, on="id_categoria", how="left") \
    .groupBy("nome_categoria") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

In [0]:
jdbc_hostname = os.getenv("SQL_HOST")
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

df_produtos.write \
    .format("sqlserver") \
    .option("host", jdbc_hostname) \
    .option("port", "1433") \
    .option("database", jdbc_database) \
    .option("dbtable", "squad1.ecommerce_produtos") \
    .option("user", jdbc_username) \
    .option("password", jdbc_password) \
    .option("encrypt", "true") \
    .mode("overwrite") \
    .save()
print("Tabela squad1.ecommerce_produtos gravada com sucesso.")

df_categorias.write \
    .format("sqlserver") \
    .option("host", jdbc_hostname) \
    .option("port", "1433") \
    .option("database", jdbc_database) \
    .option("dbtable", "squad1.ecommerce_categorias") \
    .option("user", jdbc_username) \
    .option("password", jdbc_password) \
    .option("encrypt", "true") \
    .mode("overwrite") \
    .save()
print("Tabela squad1.ecommerce_categorias gravada com sucesso.")